In [ ]:
import warnings
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, fetch_openml,load_iris,fetch_california_housing
from sklearn.feature_selection import mutual_info_regression, f_regression, RFE, SelectFromModel, SelectKBest, f_classif
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import (
RepeatedStratifiedKFold, 
cross_val_score, 
train_test_split, 
GridSearchCV,
cross_val_predict, 
learning_curve, 
validation_curve)
from sklearn.linear_model import LinearRegression,Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error,zero_one_loss, roc_auc_score,root_mean_squared_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.decomposition import PCA
from sklearn.datasets import make_circles, make_moons, make_blobs
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report, mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
from os.path import join as pjoin
from sklearn.feature_extraction.text import TfidfVectorizer
#sharper plots
%config InlineBackend.figure_format = 'retina'
from sklearn.preprocessing import MinMaxScaler
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

from sklearn.linear_model import (LogisticRegression, LogisticRegressionCV,
                                  SGDClassifier)

warnings.filterwarnings("ignore")


# Методы интерпретации
Существует множество способов разделить методы интерпретации. Например, интерпретируемые по модели, когда интерпретация заложена в структуру, или методы черного ящика, которые применяются уже после обучения модели (например, когда модель слишком сложная).

Кроме того, методы можно поделить на локальные - когда мы пытаемся объяснить предсказание для одного объекта, и глобальные - когда модель анализируется в целом.

Еще один вариант - разделение по  предназначению - применяются методы к конкретным классим модели или универсальные. Например, transformer circuits -  model specific методы.

Частично мы уже обсудили интерпретацию - рассмотрели интерпретируемые по определению модели (линейную  и дерево), а также методы для оценки важности признаков. Как можно классифицировать те методы?

# Графические методы

Наиболее информативная часть данных методов сосредоточена в графиках. Разберем три из них - ICE, PDP и ALE.

Для демонстрации методов будем использовать датасет MAGIC (Major Atmospheric Gamma Imaging Cherenkov). Это датасет для бинарной классификации гамма-лучей (signal) и адронов (background).

In [ ]:

data_path = "D:/data/ml"
columns = np.array(["Length", "Width", "Size", "Conc", "Conc1", "Asym", "M3Long", "M3Trans", "Alpha", "Dist"])

data = pd.read_csv(pjoin(data_path, "magic.txt"), header=None, names=list(columns)+["Label"])

X = data[columns].values
y = 1 * (data['Label'].values == "g")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")
print(f"Доля класса gamma (signal): {y_train.mean():.3f}")


# Масштабирование
sc = MinMaxScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

## ICE

**Individual Conditional Expectation (ICE)** - графики индивидуального условного ожидания. В этом методе мы оцениваем, как меняется предсказание для изменения конкретного признака для каждого объекта в некотором наборе $x_{test}$.

Алгоритм:
1) Зафиксируем множество $X_{test}$ и некоторый признак $j$. Пусть $j$ имеет $m$ уникальных значений $[j_1, j_2, \ldots, j_m]$.
2) Исходный датасет $X_{test}$ дублируем $m$ раз: 
    $[X'_1, X'_2, \ldots, X'_m] $
    так, что для датасета $X'_i$ значение признака $j$ фиксируется равным $j_i$.
3) На каждом $X'_i$ рассчитываем прогноз модели $f(X'_i)$.
4) На графике строим линии для каждого объекта, показывающие, 
    как прогноз (ось $y$) меняется при изменении признака (ось $x$).

**Центрированные ICE (c-ICE):**
Для лучшей визуализации часто используют центрированные ICE кривые, которые начинаются с нуля:

$$f_i^{centered}(x_S) = f_i(x_S) - f_i(x_{S,min})$$

где $x_{S,min}$ - минимальное значение признака $S$ в обучающей выборке.

ICE реализован в sklearn. Воспользуемся им.

In [ ]:

from sklearn.inspection import PartialDependenceDisplay


feature_idx = 3

PartialDependenceDisplay.from_estimator(rf, X_test, [feature_idx],
   kind='individual')

Хотя ICE очень доступны и понятны, они обладают рядом ограничений:

- Построение ICE использует предположение о независимости признаков;
- В случае большого количества объектов в наборе данных график ICE оказывается нечитаемым. Иногда ICE становится читаемее и информативнее, если раскрашивать кривые по, например, уровням какого-то признака или кластерам объектов.
- ICE не дает централизованной оценки, которую можно измерить. У этой проблемы есть решение — рассмотрение ICE в паре c PDP.

## Partial Dependence Plots (PDP)

**Partial Dependence Plot (PDP)** показывает среднее влияние признака на предсказание модели, усредняя по всем остальным признакам.

Математически PDP для признака $S$ определяется как:

$$f_S(x_S) = E_{X_C}[f(x_S, X_C)] = \int f(x_S, x_C) dP(x_C)$$

где:
- $x_S$ - значение интересующего признака (признаков)
- $X_C$ - остальные признаки
- $f$ - функция модели
- $P(x_C)$ - маргинальное распределение остальных признаков

**Алгоритм вычисления PDP:**

1. Для каждого значения $x_S$ из интересующего диапазона:
   - Зафиксировать $x_S$
   - Для каждого объекта из обучающей выборки заменить значение признака $S$ на $x_S$
   - Вычислить предсказания модели для всех модифицированных объектов
   - Усреднить предсказания

Partial Dependence plot является логическим продолжением ICE, а именно — усредняет его оценку. Это глобальная интерпретация и она также реализована в sklearn, в том числе в двумерном виде.

In [ ]:
feature_idx_alpha = list(columns).index("Alpha")
feature_idx_length = list(columns).index("Length")
feature_idx_width = list(columns).index("Width")

pdp_alpha = partial_dependence(
    rf, X_train, features=[feature_idx_alpha],
    grid_resolution=50, kind='average'
)

pdp_length = partial_dependence(
    rf, X_train, features=[feature_idx_length],
    grid_resolution=50, kind='average'
)

pdp_width = partial_dependence(
    rf, X_train, features=[feature_idx_width],
    grid_resolution=50, kind='average'
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(pdp_alpha['grid_values'][0], pdp_alpha['average'][0], linewidth=2)
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('Partial Dependence', fontsize=12)
axes[0].set_title('PDP для признака Alpha', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(pdp_length['grid_values'][0], pdp_length['average'][0], linewidth=2, color='orange')
axes[1].set_xlabel('Length', fontsize=12)
axes[1].set_ylabel('Partial Dependence', fontsize=12)
axes[1].set_title('PDP для признака Length', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(pdp_width['grid_values'][0], pdp_width['average'][0], linewidth=2, color='green')
axes[2].set_xlabel('Width', fontsize=12)
axes[2].set_ylabel('Partial Dependence', fontsize=12)
axes[2].set_title('PDP для признака Width', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


PDP позволяет также сделать дополнительные выводы о взаимодействии признаков между собой. Полностью прямые линии говорят, что взаимодействия нет (либо один признак вообще не имеет влияния в модели). Однако PDP никак не учитывает распределение признака по группам, и если для одной группы прогноз растет, а для другой падает, мы не увидим никакого влияния. Здесь возможным решением может быть построение графиков отдельно для разных групп или предварительный анализ ICE.

Задание: Постройте графики ICE, PDP для признаков LotFrontage, GarageArea  набора данных realestate из прошлогодних практик с учетом разбиения по районам (Neighborhood) и без него. Заметна ли проблема усреднения?

Кроме того, PDP также, как и  ICE предполагает независимость признаков, поэтому может приводить к некорректным выводам. 

In [ ]:
pdp_2d = partial_dependence(
    rf, X_train, features=[(feature_idx_alpha, feature_idx_length)],
    grid_resolution=30, kind='average'
)

fig, ax = plt.subplots(figsize=(10, 8))
grid_alpha = pdp_2d['grid_values'][0]
grid_length = pdp_2d['grid_values'][1]
pdp_values = pdp_2d['average'][0]

im = ax.contourf(grid_alpha, grid_length, pdp_values, levels=20, cmap='viridis')
ax.set_xlabel('Alpha', fontsize=12)
ax.set_ylabel('Length', fontsize=12)
ax.set_title('Двумерный PDP: Alpha и Length', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Partial Dependence')
plt.show()


## Accumulated Local Effects (ALE)

**Accumulated Local Effects (ALE)** решает проблему PDP с коррелированными признаками, используя условное распределение вместо маргинального.

ALE для признака $j$ определяется как:

$$ALE_j(x) = \int_{x_{min,j}}^{x} E_{X_C|X_j=z}\left[\frac{\partial f}{\partial X_j}(z, X_C)\right] dz$$

где:
- $x_{min,j}$ - минимальное значение признака $j$
- $\frac{\partial f}{\partial X_j}$ - частная производная функции модели по признаку $j$
- Усреднение происходит по условному распределению $P(X_C | X_j = z)$

**Дискретные формулы для вычисления ALE:**

В дискретном виде алгоритм вычисления ALE выглядит следующим образом:

1. **Локальный эффект для интервала $k$**:
   $$\Delta_k = \frac{1}{N_k} \sum_{i: x_{i,j} \in [z_{k-1}, z_k]} [f(z_k, x_{C,i}) - f(z_{k-1}, x_{C,i})]$$
   
   где $N_k$ - количество объектов в интервале $[z_{k-1}, z_k]$, $x_{C,i}$ - значения остальных признаков для объекта $i$.

2. **Накопленный эффект**:
   $$ALE_k = \sum_{j=1}^{k} \Delta_j$$

3. **Центрированный ALE по среднему**:
   $$ALE_k^{centered} = ALE_k - \bar{ALE}$$
   
   где $\bar{ALE} = \frac{1}{K} \sum_{k=1}^{K} ALE_k$ - среднее значение ALE по всем интервалам.

**Алгоритм вычисления ALE:**

1. **Разбиение на интервалы**: Разделить диапазон признака $j$ на $K$ интервалов
2. **Вычисление локальных эффектов**: Для каждого интервала $[z_{k-1}, z_k]$:
   - Найти объекты, у которых значение признака $j$ попадает в интервал
   - Вычислить разницу предсказаний при изменении признака от $z_{k-1}$ до $z_k$
   - Усреднить по объектам в интервале
3. **Накопление**: Проинтегрировать (суммировать) локальные эффекты от начала диапазона
4. **Центрирование**: Вычесть среднее значение ALE из всех точек, чтобы среднее значение центрированного ALE было равно нулю. Это облегчает интерпретацию и сравнение эффектов разных признаков.

In [ ]:
def compute_ale(model, X: np.ndarray, feature_idx: int, n_bins: int = 50, n_samples: int = 1000, centered: bool = True) -> tuple[np.ndarray, np.ndarray]:
    feature_values = X[:, feature_idx]
    min_val = feature_values.min()
    max_val = feature_values.max()
    
    quantiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(feature_values, quantiles)
    bin_edges[0] = min_val
    bin_edges[-1] = max_val
    
    ale_values = np.zeros(n_bins)
    ale_grid = np.zeros(n_bins)
    
    X_sample = X[np.random.choice(X.shape[0], min(n_samples, X.shape[0]), replace=False)]
    
    for k in range(n_bins):
        z_low = bin_edges[k]
        z_high = bin_edges[k + 1]
        ale_grid[k] = (z_low + z_high) / 2
        
        mask = (feature_values >= z_low) & (feature_values < z_high)
        if k == n_bins - 1:
            mask = (feature_values >= z_low) & (feature_values <= z_high)
        
        if mask.sum() == 0:
            continue
        
        X_low = X_sample.copy()
        X_high = X_sample.copy()
        X_low[:, feature_idx] = z_low
        X_high[:, feature_idx] = z_high
        
        pred_low = model.predict_proba(X_low)[:, 1]
        pred_high = model.predict_proba(X_high)[:, 1]
        
        local_effect = np.mean(pred_high - pred_low)
        ale_values[k] = local_effect
    
    ale_accumulated = np.cumsum(ale_values)
    
    if centered:
        ale_accumulated = ale_accumulated - np.mean(ale_accumulated)
    
    return ale_grid, ale_accumulated


In [ ]:
ale_alpha_grid, ale_alpha_values = compute_ale(rf, X_train, feature_idx_alpha, n_bins=50)
ale_length_grid, ale_length_values = compute_ale(rf, X_train, feature_idx_length, n_bins=50)
ale_width_grid, ale_width_values = compute_ale(rf, X_train, feature_idx_width, n_bins=50)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(ale_alpha_grid, ale_alpha_values, linewidth=2, color='blue', label='ALE')
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('ALE Value', fontsize=12)
axes[0].set_title('ALE для признака Alpha', fontsize=14, fontweight='bold')
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(ale_length_grid, ale_length_values, linewidth=2, color='orange', label='ALE')
axes[1].set_xlabel('Length', fontsize=12)
axes[1].set_ylabel('ALE Value', fontsize=12)
axes[1].set_title('ALE для признака Length', fontsize=14, fontweight='bold')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(ale_width_grid, ale_width_values, linewidth=2, color='green', label='ALE')
axes[2].set_xlabel('Width', fontsize=12)
axes[2].set_ylabel('ALE Value', fontsize=12)
axes[2].set_title('ALE для признака Width', fontsize=14, fontweight='bold')
axes[2].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
ale_alpha_grid_centered, ale_alpha_values_centered = compute_ale(rf, X_train, feature_idx_alpha, n_bins=50, centered=True)
ale_length_grid_centered, ale_length_values_centered = compute_ale(rf, X_train, feature_idx_length, n_bins=50, centered=True)
ale_width_grid_centered, ale_width_values_centered = compute_ale(rf, X_train, feature_idx_width, n_bins=50, centered=True)

ale_alpha_grid_uncentered, ale_alpha_values_uncentered = compute_ale(rf, X_train, feature_idx_alpha, n_bins=50, centered=False)
ale_length_grid_uncentered, ale_length_values_uncentered = compute_ale(rf, X_train, feature_idx_length, n_bins=50, centered=False)
ale_width_grid_uncentered, ale_width_values_uncentered = compute_ale(rf, X_train, feature_idx_width, n_bins=50, centered=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(ale_alpha_grid_centered, ale_alpha_values_centered, linewidth=2, color='blue', label='ALE (центрированное)')
axes[0, 0].set_xlabel('Alpha', fontsize=12)
axes[0, 0].set_ylabel('ALE Value', fontsize=12)
axes[0, 0].set_title('ALE для признака Alpha (центрированное)', fontsize=14, fontweight='bold')
axes[0, 0].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(ale_length_grid_centered, ale_length_values_centered, linewidth=2, color='orange', label='ALE (центрированное)')
axes[0, 1].set_xlabel('Length', fontsize=12)
axes[0, 1].set_ylabel('ALE Value', fontsize=12)
axes[0, 1].set_title('ALE для признака Length (центрированное)', fontsize=14, fontweight='bold')
axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[0, 2].plot(ale_width_grid_centered, ale_width_values_centered, linewidth=2, color='green', label='ALE (центрированное)')
axes[0, 2].set_xlabel('Width', fontsize=12)
axes[0, 2].set_ylabel('ALE Value', fontsize=12)
axes[0, 2].set_title('ALE для признака Width (центрированное)', fontsize=14, fontweight='bold')
axes[0, 2].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].legend()

axes[1, 0].plot(ale_alpha_grid_uncentered, ale_alpha_values_uncentered, linewidth=2, color='blue', label='ALE (нецентрированное)')
axes[1, 0].set_xlabel('Alpha', fontsize=12)
axes[1, 0].set_ylabel('ALE Value', fontsize=12)
axes[1, 0].set_title('ALE для признака Alpha (нецентрированное)', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(ale_length_grid_uncentered, ale_length_values_uncentered, linewidth=2, color='orange', label='ALE (нецентрированное)')
axes[1, 1].set_xlabel('Length', fontsize=12)
axes[1, 1].set_ylabel('ALE Value', fontsize=12)
axes[1, 1].set_title('ALE для признака Length (нецентрированное)', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

axes[1, 2].plot(ale_width_grid_uncentered, ale_width_values_uncentered, linewidth=2, color='green', label='ALE (нецентрированное)')
axes[1, 2].set_xlabel('Width', fontsize=12)
axes[1, 2].set_ylabel('ALE Value', fontsize=12)
axes[1, 2].set_title('ALE для признака Width (нецентрированное)', fontsize=14, fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# !pip install -qU pyALE;

In [ ]:
from PyALE import ale

## 1D - continuous - no CI
ale_eff = ale(
    X=pd.DataFrame(X_test,columns=columns), model=rf, feature=['Length'], grid_size=15,
)

In [ ]:
## 1D - continuous - no CI
ale_eff = ale(
    X=pd.DataFrame(X_test,columns=columns), model=rf, feature=['Alpha'], grid_size=15,
)

In [ ]:
## 1D - continuous - no CI
ale_eff = ale(
    X=pd.DataFrame(X_test,columns=columns), model=rf, feature=['Width'], grid_size=15,
)

# LIME
Рассмотрим метод LIME (Local Interpretable Model Agnostic explanations), который позволяет построить интерпретацию для некоторого объясняемого объекта x* и его окрестности.
Для этого независимо от модели $a(x)$ для объекта вводится его объясняемое представление, и обучается модель $\hat a(x)$, которая строит предсказания для этих признаков. Чаще всего для этого используются довольно простые представления (например, мешок слов или суперпиксели). 
Для построения объясняющей модели:
1) Строится суррогатная выборка $X_{x*}^l = {(\hat x_i, y_i)}$ :
    1) создается l примеров $\hat x_i$ путем пертурбации исходных признаков, например, обнуления случайного количества единиц в интерпретируемом представлении $\hat x*$ объекта для текстов или  картинок. Вопрос, как в примере с текстом проходит обнуление?
    3) для каждого $\hat x_i$ делается переход в исходное пространство 
    4) получается $y = a(x_i)$

2) Обучается объясняющая модель:
   $$a_i = argmin_{b \in B} {\sum_{x_i} {\pi_{x*}(x_i)(a(x_i)-b((\hat x_i))^2+\sigma(b)}},$$
    - $B$ - cемейство объясняющих моделей, $\sigma(b)$ - ее сложность
    - $\pi_{x*}(x_i)$ - вес объекта, полученный с помощью некоторого ядра

Таким образом, объекты, близкие к x*, получают больший вес при обучении bб а удаленные - меньший. 

В качестве функции ошибки можно использовать и другую какую-то.
Пример - разреженные линейные представления (SLE), если использовать квадратичную функцию и $\sigma(b) = \infty[||w_b||_0 K]$ (Что за норма такая $||w_b||_0$?). Для интерпретации в таком случае достаточно вывести признаки с ненулевыми весами - так мы увидим, какие слова были использованы как самые "важные"

В итоге мы получаем довольно неплохую локальную интерпретацию, но глобально она будет, конечно же, не очень.

На практике обычно оптимизируется только часть, относящаяся к лоссу, а сложность контролируется за счет регуляризации объясняющей модели или RFE. 
Как это делается (для регуляризации): Задается большое $\lambda$, после чего оно постепенно уменьшается, пока не будет достигнуто K признаков с ненулевыми весами.

$\pi_{x*}(x_i)$ представляет собой функцию расстояния. Чаще всего используется гауссово ядро: 

$\pi_{x*}(x_i) = exp({-{d(x*, x_i)^2} \over {2\sigma^2}})$

Тут $\sigma$ - ширина ядра. Ее следует выбрать так, чтобы не размывать локальный контекст, но и не игнорировать информацию в дальних примерах.

## Таблицы

Как LIME работает с таблицами?

Во-первых, используются исходные признаки, не бинарные представления.

Во-вторых, выборки строятся путем вытягивания значения из нормального распределения со средним и стандартным отклонением, полученными из обучающего набора для каждой фичи независимо.
Кажется, что это не поможет нам строить интерпретацию для конкретного семпла. Но это работает, по крайней мере, частично. Вопрос: почему?

Для демонстрации используем уже знакомый нам датасет про дома

In [ ]:
data_path = r"d:\data\ml" 

In [ ]:
data = pd.read_csv(pjoin(data_path, "realestate.txt"), sep="\t")
data.fillna(data.median(), inplace=True)
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
y.mean(), y.min(), y.max(), y.std()

In [ ]:
data.columns

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(max_depth=5).fit(X_train, y_train) # Обучите случайный лес 


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
model.score(X_test, y_test)
root_mean_squared_error(y_pred, y_test)

In [ ]:
import lime
import lime.lime_tabular

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values, feature_names=X_train.columns.values.tolist(),
                                                  class_names=['SalePrice'], verbose=True, mode='regression', discretize_continuous=True)

In [ ]:
j = 5
exp = explainer.explain_instance(X_test.values[j], model.predict, num_features=6)

In [ ]:
exp.show_in_notebook(show_table=True)

Здесь положительное влияние показано справа, отрицательное - слева. Значения соответствуют весам линейной модели, аппроксимирующей наш случайный лес. Грубо говоря, если мы уменьшим year на 12, то и наше предсказание уменьшится на 31

Проверим, так ли

In [ ]:
X_test.values[j]

In [ ]:
print('Original prediction:', model.predict(X_test.values[j].reshape(1, -1)))
tmp = X_test.values[j].copy()
tmp[6] = 1980

In [ ]:
print('Prediction removing some features:',  model.predict(tmp.reshape(1, -1)))
print('Difference:', model.predict(tmp.reshape(1, -1)) - model.predict(X_test.values[j].reshape(1, -1)))

In [ ]:
exp.show_in_notebook(show_table=True)

Интересно, что для категориальных фичей мы снова можем построить более удачные объяснения. Как мы поняли, LIME требует бинарные представления. Как мы можем к ним придти? 

In [ ]:
X_train.head()

Мы можем задать, какие фичи у нас категориальные. 

In [ ]:
categorical_features = ["Beds", "Baths", "Air", "Garage", "Pool", "Quality", "Style", "Highway"]
categorical_feature_ids = [1,2,3,4,5,7,8,10]

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, 
    feature_names=X_train.columns.values.tolist(),
    class_names=['SalePrice'], 
    categorical_features=categorical_feature_ids,
    categorical_names=categorical_features,
    verbose=True,
    mode='regression', 
    discretize_continuous=True
)
# Объясните тот же пример

Можно вообще извратиться и сделать за LIME бинаризацию. 

In [ ]:
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoder.fit(X_train[categorical_features])
cat_df = pd.DataFrame(
    encoder.transform(X_train[categorical_features]),
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
X_train_ohe = pd.concat([X_train[["SqFeet","Year", "Lot"]], cat_df], axis=1)

In [ ]:
X_train_ohe.head()

In [ ]:
cat_df = pd.DataFrame(
    encoder.transform(X_test[categorical_features]),
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)
X_test_ohe = pd.concat([X_test[["SqFeet","Year", "Lot"]], cat_df], axis=1)

In [ ]:
encoder.get_feature_names_out()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train_ohe, y_train)

In [ ]:
# Настройте explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_ohe.values, 
    feature_names=X_train_ohe.columns.values.tolist(),
    class_names=['SalePrice'], 
    verbose=True, 
    mode='regression', 
    discretize_continuous=True
)

In [ ]:
# Постройте интерпретацию для 5 семпла (id = 4)
j = 5
exp = explainer.explain_instance(X_train_ohe.values[j], model.predict, num_features=8)

In [ ]:
exp.show_in_notebook(show_table=True)

Мы можем также задать ширину ядра и его вид.

**Задание**: Напишите свои ядра, берущие на вход расстояния и возвращающие значения от 0 до 1. Замените ядро по умолчанию. 

## Тексты

Также рассмотрим интерпретацию для классификации новостей на две группы - атеизм и христианство.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
categories = ['alt.atheism', 'soc.religion.christian']
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories)
class_names = ['atheism', 'christian']

Векторизуем тексты с помощью TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(lowercase=False)
train_vectors = vectorizer.fit_transform(newsgroups_train.data)
test_vectors =  vectorizer.transform(newsgroups_test.data)# сделайте то же самое и для теста

In [ ]:
rf = RandomForestClassifier().fit(train_vectors, newsgroups_train.target) # Обучите случайный лес

In [ ]:
pred = rf.predict(test_vectors)
f1_score(newsgroups_test.target, pred, average='binary')

In [ ]:
from sklearn.pipeline import make_pipeline
c = make_pipeline(vectorizer, rf)

In [ ]:
print(c.predict_proba([newsgroups_test.data[0]]))

In [ ]:
from lime.lime_text import LimeTextExplainer
explainer = LimeTextExplainer(class_names=class_names)

In [ ]:
idx = 83
exp = explainer.explain_instance(newsgroups_test.data[idx], c.predict_proba, num_features=6)
print('Document id: %d' % idx)
print('Probability(christian) =', c.predict_proba([newsgroups_test.data[idx]])[0,1])
print('True class: %s' % class_names[newsgroups_test.target[idx]])

In [ ]:
exp.as_list()

In [ ]:
exp.show_in_notebook(text=False)

Посмотрим, сохраняется ли ситуация с изменением фичей. Тут ожидание следующее - вероятность увеличится на 0.3, если мы уберем из текста слова Host и Posting.

In [ ]:
print('Original prediction:', rf.predict_proba(test_vectors[idx])[0,1])
tmp = test_vectors[idx].copy()
tmp[0,vectorizer.vocabulary_['Posting']] = 0
tmp[0,vectorizer.vocabulary_['Host']] = 0
print('Prediction removing some features:', rf.predict_proba(tmp)[0,1])
print('Difference:', rf.predict_proba(tmp)[0,1] - rf.predict_proba(test_vectors[idx])[0,1])

Для текстов мы можем еще и посмотреть на самые важные слова:

In [ ]:
exp.show_in_notebook(text=True)

Судя по всему, на самом деле моделька смотрит на мусор типа заголовков или частых слов. Проверим, так ли это для какого-то другого примера:

In [ ]:
idx = 34
exp = explainer.explain_instance(newsgroups_test.data[idx], c.predict_proba, num_features=6)
print('Document id: %d' % idx)
print('Probability(christian) =', c.predict_proba([newsgroups_test.data[idx]])[0,1])
print('True class: %s' % class_names[newsgroups_test.target[idx]])
exp.show_in_notebook(text=True)

Плюсы: 
- Можно поменять предсказывающую модель, но все еще использовать ту же объясняющую
- Легко применить, работает с разными видами данных
  
Минусы:
- Для табличных данных очень сложно выбрать ядро, да и вообще выбор ядра может значительно поменять интерпретацию.
- Сложность объяснения должна быть выбрана заранее
- Нестабильный

**Задание(*):**
У вас есть все составляющие LIME - ядра, алгоритм, кодировка фичей, loess с прошлой пары. Не хватает только кода с семплированием.
Напишите свой класс Lime для регрессии и таблиц, который с помощью линейной регрессии и заданного ядра строит интерпретацию. В качестве сложности используйте способ с регуляризацией (можете LARS). При вызове он должен вернуть коэффициенты регрессии, использованные для интерпретации. Не забудьте отшкалиировать фичи. В качестве метрики используйте евклидову.
Сделайте упрощенное семплирование - только нормальное распределение и дискретное по частотности категорий.

In [ ]:
class LIME:
    def __init__(self, train_df, kernel, kernel_width, categorical_features=None):
        pass

    def _generate_sample(self, instance):
        pass
        
    def explain(self, instance, prediction, num_features=5):
        pass
        

## SHAP

Значения Шепли как практический инструмент решают три проблемы сразу:
- оценка локального влияния прищнака на прогноз
- оценка глобального влияния признаков
- оценка взаимодействия признаков

За счет универсальности и согласованности, а также независимости от модели, это один из наиболее популярных методов интрепретации признаков. 

Изначально этот метод пришел из теории игр, поэтому нам нужны дополнительные определения:

1. **Коалиционная игра** - это пара $(N, v)$, где:
   - $N = \{1, 2, \ldots, M\}$ - множество игроков (в нашем случае - множество признаков)
   - $v: 2^N \rightarrow \mathbb{R}$ - характеристическая функция, которая ставит в соответствие каждой коалиции (подмножеству признаков) $S \subseteq N$ некоторое значение $v(S)$. В контексте машинного обучения это функция, которая возвращает предсказание модели при использовании только признаков из подмножества $S$:
   $$v(S) = E[f(X) | X_S = x_S]$$
   где $X_S$ - признаки из подмножества $S$, а $x_S$ - их конкретные значения для объясняемого объекта.

3. **Маргинальный вклад признака $i$** при добавлении к коалиции $S$:
   $$\Delta_i(S) = v(S \cup \{i\}) - v(S)$$
   Это разница между предсказанием модели с признаком $i$ и без него, при фиксированных значениях признаков из $S$.

**Формула значения Шепли:**

Значение Шепли для признака $i$ определяется как средневзвешенный маргинальный вклад по всем возможным коалициям:

$$\phi_i(v) = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(M - |S| - 1)!}{M!} [v(S \cup \{i\}) - v(S)]$$

где:
- $M$ - общее количество признаков
- $S$ - подмножество признаков без признака $i$
- $|S|$ - размер подмножества $S$
- $\frac{|S|!(M - |S| - 1)!}{M!}$ - весовой коэффициент, равный вероятности того, что признак $i$ будет добавлен к случайно выбранной коалиции размера $|S|$

**Алгоритм вычисления значения Шепли:**

1. **Для каждого признака $i \in \{1, 2, \ldots, M\}$:**
   
2. **Для каждой возможной коалиции $S \subseteq N \setminus \{i\}$:**
   - Вычислить предсказание модели $v(S)$ без признака $i$ (используя только признаки из $S$)
   - Вычислить предсказание модели $v(S \cup \{i\})$ с признаком $i$ (используя признаки из $S$ и признак $i$)
   - Вычислить маргинальный вклад: $\Delta_i(S) = v(S \cup \{i\}) - v(S)$
   - Вычислить вес коалиции: $w(S) = \frac{|S|!(M - |S| - 1)!}{M!}$

3. **Усреднить маргинальные вклады с весами:**
   $$\phi_i = \sum_{S \subseteq N \setminus \{i\}} w(S) \cdot \Delta_i(S)$$


**Свойства значений Шепли (аксиомы):**

1. **Эффективность (Efficiency)**: $\sum_{i=1}^{M} \phi_i = v(N) - v(\emptyset)$
   - Сумма всех значений Шепли равна разнице между предсказанием со всеми признаками и предсказанием без признаков

2. **Симметрия (Symmetry)**: Если $v(S \cup \{i\}) = v(S \cup \{j\})$ для всех $S$, то $\phi_i = \phi_j$
   - Признаки с одинаковым влиянием получают одинаковые значения

3. **Фиктивность (Dummy)**: Если $v(S \cup \{i\}) = v(S)$ для всех $S$, то $\phi_i = 0$
   - Признак, не влияющий на предсказание, получает нулевое значение

4. **Аддитивность (Additivity)**: Для двух игр $v$ и $w$: $\phi_i(v + w) = \phi_i(v) + \phi_i(w)$
   - Значения Шепли аддитивны относительно комбинации игр



In [ ]:
import shap
#Локальное объяснение с shap

shap.initjs()
explainer = shap.TreeExplainer(model, feature_names=X_test_ohe.columns)
shap_values = explainer(X_test_ohe)

X_train_sc_df = pd.DataFrame(X_test_ohe, columns=X_test_ohe.columns)


Force Plot показывает локальное объяснение предсказания модели для одного объекта. На этом графике:
- Базовое значение (base value) - среднее предсказание модели на обучающей выборке.
- Ширина полосы отображает вклад.
- Финальное предсказание (output value) - результат суммирования базового значения и всех вкладов:
   $$f(x) = \text{base\_value} + \sum_{i=1}^{M} \phi_i$$
Признаки обычно отсортированы по величине вклада (по убыванию абсолютного значения).Вклад считается только для конкртеного семпла иотличается от глобальной важности (и можеет даже не совпадать по порядку с общей важностью)

Для классификации можно использовать аргумент link="logit", чтобы выводить логиты. Кроме того, для классификации удобно использовать другой стиль для неправильно отклассифицированных семплов (например через  highlight=misclassified)

In [ ]:
# Визуализируем прогнозы для 5-го объекта в тестовых данных
shap.plots.force(shap_values[5, :])

Decision Plot показывает, как модель "принимает решение" для одного или нескольких объектов, показывая путь от базового значения к финальному предсказанию.
Ось Y - список признаков, отсортированных по важности (сверху вниз - от самых важных к менее важным)
Ось X - значения предсказания модели (обычно в шкале исходной задачи - например, цена дома или вероятность класса)
Каждая линия представляет один объект:
   - Линия начинается слева от базового значения (expected value)
   - Проходит через все признаки, накапливая их вклады
   - Заканчивается справа на финальном предсказании для этого объекта


In [ ]:
shap.decision_plot(
    explainer.expected_value, 
    explainer.shap_values(X_test_ohe[5:6]), 
    X_test_ohe[5:6]
)

Decision plot бывает удобнее force plot, например, когда число фичей велико и хочется явно увидеть их вклады. Кроме того, этот график поддерживает несколько выходов модели или позволяет сравнить несколько моделей.

In [ ]:
shap_interaction_values = explainer.shap_interaction_values(X_test_ohe[5:6])
shap.decision_plot(
    explainer.expected_value,
    shap_interaction_values,
    X_test_ohe[5:6]
)

С помощью этого графика можно также находить выбросы (если линии сильно отклоняются от стандартных), оценить гипотетические сценарии (что будет, если площадь дома изменится на 100 метров)

In [ ]:
idx = 5
n_scenarios = 10
target_col = 'SqFeet'
original_sample = X_test_ohe.iloc[idx: idx+1].copy()

# Создаем выборку, заменяя значение признака на выбранную из интервала (+- половина всего интервала значений столбца)
original_val = original_sample[target_col].values[0]

val_min = X_train[target_col].min()
val_max = X_train[target_col].max()
val_range = val_max - val_min

values = np.linspace(
    max(val_min, original_val - 0.5 * val_range),
    min(val_max, original_val + 0.5 * val_range),
    n_scenarios
)

hypothetical_samples = []
for val in values:
    sample = original_sample.copy()
    sample[target_col] = val
    hypothetical_samples.append(sample)

hypothetical_df = pd.concat(hypothetical_samples, ignore_index=True)

all_samples = pd.concat([original_sample, hypothetical_df], ignore_index=True)
all_shap_values = explainer.shap_values(all_samples)

original_idx_in_all = 0

shap.decision_plot(
    explainer.expected_value,
    all_shap_values,
    all_samples,
    highlight=[original_idx_in_all],
    feature_order="hclust", #  features are ordered via hierarchical clustering to group similar prediction paths 
)


Перейдем к глобальной интерпретации. Ее можно провести, построив общий график, где каждая точка будет соответствовать одному семплу.

In [ ]:
X_train_sc.shape # очень много семплов!

In [ ]:
shap.dependence_plot("SqFeet", shap_values.values[:100], X_train_sc_df[:100], interaction_index="Year")

In [ ]:
# глобальное поведение через force plot
shap.initjs()

#Посмотрим на график для первых 100 объектов тренировочного набора данных
shap.plots.force(shap_values[:100])

На графике — наблюдения, отсортированные в выбранном порядке (панель сверху). По умолчанию наблюдения остортированы по близости друг к другу (и по оси x — номер наблюдения в отсортированном ряду), а по оси y располагается прогноз модели.

Для y отображения можно выбирать различные признаки — и тогда будет видно, как конкретный признак менял output модели.

Иногда мы не знаем заранее, какой признак мы хотим вывести. Поэтому можно отрисовать самый важный по мнению Shap. В таком случае, мы хотим узнать глобальную важность признака как среднее значение shap и отсортировать признаки по этому значению (в целом, мы так в принципе можем автоматизировать выбор признака). Заодно посмотри на более простой график значений shap.

In [ ]:
shap.plots.scatter(shap_values[:100,  shap_values.abs.mean(0).argsort[-1]])

In [ ]:
# Второй по важности
shap.plots.scatter(shap_values[:100,  shap_values.abs.mean(0).argsort[-2]])

Мы можем также и узнать взаимодействие двух признаков - отрисовать значения для 1 признака, раскрасив график с помощью второго:


In [ ]:
shap.plots.scatter(shap_values[:100,  shap_values.abs.mean(0).argsort[-1]], color = shap_values[:100,  shap_values.abs.mean(0).argsort[-2]])

**Задание:** постройте интерпретацию и для других признаков (хотя бы пары). Сделайте выводы об их влиянии на прогноз в целом и нескольких примеров.

Основной недостаток метода - вычислительная сложность, так как для n признаков необходимо перебрать $2^n$ подмножеств. Для больших n используются приближенные алгоритмы (KernelShap, TreeShap).